# Formateo de Archivos de Votaciones
---

Este notebook procesa los archivos CSV de votaciones de diputados y los formatea para análisis posterior

## Objetivos

* Leer múltiples archivos CSV de votaciones
* Normalizar nombres de columnas
* Asignar IDs únicos a legisladores, bloques y provincias
* Exportar datos consolidados

In [1]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
from datetime import datetime
import glob
import unicodedata
from tqdm import tqdm

In [3]:
# Definir rutas de trabajo
carpeta_legislativas = 'legislativas/'
carpeta_votaciones = os.path.join(carpeta_legislativas, "diputados", "01_scraping", "votaciones_csvs")
carpeta_sesiones = os.path.join(carpeta_legislativas, "diputados", "01_scraping", "sesiones")

# Crear carpeta de sesiones si no existe
os.makedirs(carpeta_sesiones, exist_ok=True)

In [4]:
# Función para normalizar texto (eliminar acentos y convertir a minúsculas)
def normalizar_texto(texto):
    """
    Normaliza texto eliminando acentos y convirtiendo a minúsculas.
    Equivalente a stringi::stri_trans_general(tolower(text), 'Latin-ASCII') en R
    """
    if pd.isna(texto):
        return texto
    # Normalizar unicode (NFD = descomponer caracteres con acentos)
    texto_nfd = unicodedata.normalize('NFD', str(texto).lower())
    # Filtrar solo caracteres ASCII
    texto_ascii = ''.join(char for char in texto_nfd if unicodedata.category(char) != 'Mn')
    return texto_ascii

In [5]:
# Listar archivos de votaciones que empiezan con 'votacion_' seguido de 4 dígitos
patron_votacion = os.path.join(carpeta_votaciones, "votacion_*.csv")
archivos_sesiones = sorted(glob.glob(patron_votacion))

# Filtrar solo los que tienen el patrón correcto (votacion_NNNN_...)
archivos_sesiones = [f for f in archivos_sesiones if re.search(r'votacion_\d{4}', os.path.basename(f))]

print(f"Total de archivos de votaciones encontrados: {len(archivos_sesiones)}")

Total de archivos de votaciones encontrados: 395


In [8]:
# Procesar todos los archivos de votaciones
sesiones = []

for archivo in tqdm(archivos_sesiones, desc="Cargando sesiones"):
    # Extraer número de sesión del nombre del archivo
    nombre_archivo = os.path.basename(archivo)
    match = re.search(r'votacion_(\d{4})', nombre_archivo)
    
    if match:
        numero_sesion = int(match.group(1))
        
        # Leer el CSV
        try:
            sesion = pd.read_csv(archivo, encoding='utf-8')
            
            # Agregar ID de votación
            sesion['id_votacion'] = numero_sesion
            
            # Renombrar columnas (equivalente a setnames en R)
            columnas_map = {
                'DIPUTADO': 'legislador',
                'BLOQUE': 'bloque',
                'PROVINCIA': 'provincia',
                '¿CÓMO VOTÓ?': 'voto'
            }
            sesion = sesion.rename(columns=columnas_map)
            
            sesiones.append(sesion)
        except Exception as e:
            print(f"Error procesando {nombre_archivo}: {e}")

# Concatenar todas las sesiones (equivalente a rbindlist en R)
sesiones_dt = pd.concat(sesiones, ignore_index=True)
sesiones_dt.head()

Cargando sesiones: 100%|██████████| 395/395 [00:00<00:00, 578.64it/s]


,legislador,bloque,provincia,voto,id_votacion
0,"AGUIRRE, HILDA",Frente De Todos,La Rioja,AFIRMATIVO,4331
1,"AGUIRRE, MANUEL IGNACIO",Unión Cívica Radical,Corrientes,AFIRMATIVO,4331
2,"AJMECHET, SABRINA",Pro,C.A.B.A.,AFIRMATIVO,4331
3,"ALDERETE, JUAN CARLOS",Frente De Todos,Buenos Aires,NEGATIVO,4331
4,"ALIANIELLO, EUGENIA",Frente De Todos,Chubut,AFIRMATIVO,4331


In [9]:
# Cargar información de sesiones
archivo_info_sesiones = glob.glob(os.path.join(carpeta_votaciones, "votaciones_hcdn_main*.csv"))[0]
info_sesiones = pd.read_csv(archivo_info_sesiones)
info_sesiones.head()

,fecha,titulo,tipo,resultado,id_votacion
0,17/09/2025 18:37,HABILITACIÓN DEL TRATAMIENTO EXPTE. 32-S-2025 ...,Votación Nominal,NEGATIVO,5768
1,17/09/2025 17:32,"MOCIÓN SOLICITADA POR EL DIP. DE LOREDO, RODRIGO.",Votación Nominal,NEGATIVO,5765
2,17/09/2025 17:14,INSISTENCIA PROYECTO DE LEY 27.795.,Votación Nominal,AFIRMATIVO,5766
3,17/09/2025 17:09,INSISTENCIA PROYECTO DE LEY 27.796.,Votación Nominal,AFIRMATIVO,5762
4,17/09/2025 14:01,HABILITACIÓN DEL TRATAMIENTO EXPTE. 9-PE-2025.,Votación Nominal,AFIRMATIVO,5761


In [10]:
# Procesar fechas y extraer año (equivalente a dmy_hm y year en R)
info_sesiones['fecha_hora'] = pd.to_datetime(info_sesiones['fecha'], format='%d/%m/%Y %H:%M')
info_sesiones['yr'] = info_sesiones['fecha_hora'].dt.year

print("Años disponibles en las sesiones:")
print(info_sesiones['yr'].value_counts().sort_index())

Años disponibles en las sesiones:
yr
2022     79
2023     55
2024    180
2025     81
Name: count, dtype: int64


In [11]:
# Agregar año a las sesiones mediante merge (equivalente al join en data.table R)
sesiones_dt = sesiones_dt.merge(
    info_sesiones[['id_votacion', 'yr']], 
    on='id_votacion', 
    how='left'
)

print(f"Años agregados a sesiones_dt:")
print(sesiones_dt['yr'].value_counts().sort_index())

Años agregados a sesiones_dt:
yr
2022    20303
2023    14135
2024    46260
2025    20817
Name: count, dtype: int64


## Normalización y Asignación de IDs

Ahora normalizaremos los nombres de legisladores, bloques y provincias, y asignaremos IDs únicos a cada uno

In [12]:
# Normalizar y asignar ID a legisladores
sesiones_dt['legislador_norm'] = sesiones_dt['legislador'].apply(normalizar_texto)

# Ordenar por nombre normalizado
sesiones_dt = sesiones_dt.sort_values('legislador_norm').reset_index(drop=True)

# Asignar ID único por legislador (equivalente a .GRP en data.table)
legisladores_unicos = sesiones_dt[['legislador_norm']].drop_duplicates().reset_index(drop=True)
legisladores_unicos['legislador_id'] = legisladores_unicos.index + 1

# Hacer merge de IDs
sesiones_dt = sesiones_dt.merge(legisladores_unicos, on='legislador_norm', how='left')

# Eliminar columna temporal
sesiones_dt = sesiones_dt.drop('legislador_norm', axis=1)

print(f"Total de legisladores únicos: {sesiones_dt['legislador_id'].nunique()}")

Total de legisladores únicos: 385


In [13]:
# Normalizar y asignar ID a bloques
sesiones_dt['bloque_norm'] = sesiones_dt['bloque'].apply(normalizar_texto)
sesiones_dt = sesiones_dt.sort_values('bloque_norm').reset_index(drop=True)

bloques_unicos = sesiones_dt[['bloque_norm']].drop_duplicates().reset_index(drop=True)
bloques_unicos['bloque_id'] = bloques_unicos.index + 1

sesiones_dt = sesiones_dt.merge(bloques_unicos, on='bloque_norm', how='left')
sesiones_dt = sesiones_dt.drop('bloque_norm', axis=1)

print(f"Total de bloques únicos: {sesiones_dt['bloque_id'].nunique()}")

Total de bloques únicos: 54


In [14]:
# Normalizar y asignar ID a provincias
sesiones_dt['provincia_norm'] = sesiones_dt['provincia'].apply(normalizar_texto)
sesiones_dt = sesiones_dt.sort_values('provincia_norm').reset_index(drop=True)

provincias_unicas = sesiones_dt[['provincia_norm']].drop_duplicates().reset_index(drop=True)
provincias_unicas['provincia_id'] = provincias_unicas.index + 1

sesiones_dt = sesiones_dt.merge(provincias_unicas, on='provincia_norm', how='left')
sesiones_dt = sesiones_dt.drop('provincia_norm', axis=1)

print(f"Total de provincias únicas: {sesiones_dt['provincia_id'].nunique()}")

Total de provincias únicas: 24


In [15]:
# Verificar el resultado final
print(sesiones_dt.info())
sesiones_dt.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101515 entries, 0 to 101514
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   legislador     101515 non-null  object
 1   bloque         101515 non-null  object
 2   provincia      101515 non-null  object
 3   voto           101515 non-null  object
 4   id_votacion    101515 non-null  int64 
 5   yr             101515 non-null  int32 
 6   legislador_id  101515 non-null  int64 
 7   bloque_id      101515 non-null  int64 
 8   provincia_id   101515 non-null  int64 
dtypes: int32(1), int64(4), object(4)
memory usage: 6.6+ MB
None


,legislador,bloque,provincia,voto,id_votacion,yr,legislador_id,bloque_id,provincia_id
0,"BESANA, GABRIELA",Pro,Buenos Aires,AFIRMATIVO,4528,2022,50,39,1
1,"SALVADOR, SEBASTIAN NICOLAS",Ucr,Buenos Aires,NEGATIVO,4589,2022,316,49,1
2,"SALVADOR, SEBASTIAN NICOLAS",Ucr,Buenos Aires,AFIRMATIVO,4649,2022,316,49,1
3,"SALVADOR, SEBASTIAN NICOLAS",Ucr,Buenos Aires,AFIRMATIVO,4594,2022,316,49,1
4,"SALVADOR, SEBASTIAN NICOLAS",Ucr,Buenos Aires,AFIRMATIVO,4828,2023,316,49,1
5,"SALVADOR, SEBASTIAN NICOLAS",Ucr,Buenos Aires,AFIRMATIVO,4656,2022,316,49,1
6,"SALVADOR, SEBASTIAN NICOLAS",Ucr,Buenos Aires,ABSTENCION,4866,2023,316,49,1
7,"SALVADOR, SEBASTIAN NICOLAS",Ucr,Buenos Aires,AFIRMATIVO,4862,2023,316,49,1
8,"SALVADOR, SEBASTIAN NICOLAS",Ucr,Buenos Aires,AFIRMATIVO,4816,2023,316,49,1
9,"SALVADOR, SEBASTIAN NICOLAS",Ucr,Buenos Aires,AFIRMATIVO,4733,2023,316,49,1


## Exportar Datos Procesados

In [16]:
# Guardar info_sesiones
ruta_info_sesiones = os.path.join(carpeta_sesiones, "info_sesiones_dt.csv")
info_sesiones.to_csv(ruta_info_sesiones, index=False)
print(f"Guardado: {ruta_info_sesiones}")

# Guardar sesiones_dt comprimido (equivalente a fwrite con .gz en R)
ruta_sesiones = os.path.join(carpeta_sesiones, "sesiones_dt.csv.gz")
sesiones_dt.to_csv(ruta_sesiones, index=False, compression='gzip')
print(f"Guardado: {ruta_sesiones}")

Guardado: legislativas/diputados/01_scraping/sesiones/info_sesiones_dt.csv
Guardado: legislativas/diputados/01_scraping/sesiones/sesiones_dt.csv.gz


In [17]:
# Resumen estadístico final
print("="*60)
print("RESUMEN ESTADÍSTICO FINAL")
print("="*60)
print(f"Total de votaciones registradas: {len(sesiones_dt):,}")
print(f"Total de sesiones: {sesiones_dt['id_votacion'].nunique():,}")
print(f"Total de legisladores únicos: {sesiones_dt['legislador_id'].nunique()}")
print(f"Total de bloques únicos: {sesiones_dt['bloque_id'].nunique()}")
print(f"Total de provincias únicas: {sesiones_dt['provincia_id'].nunique()}")
print(f"Años cubiertos: {sorted(sesiones_dt['yr'].dropna().unique().astype(int).tolist())}")
print("="*60)

RESUMEN ESTADÍSTICO FINAL
Total de votaciones registradas: 101,515
Total de sesiones: 395
Total de legisladores únicos: 385
Total de bloques únicos: 54
Total de provincias únicas: 24
Años cubiertos: [2022, 2023, 2024, 2025]
